## MÜŞTERİ SEGMENTASYONU VE CHURN TAHMİNİ

Bu bölümde, teorik altyapısı kurulan Kümeleme (K-Means), Sınıflandırma (Lojistik Regresyon) ve Düzenlileştirme (L1/L2) algoritmalarının e-ticaret sektörü özelinde entegre bir veri bilimi problemine uygulanması ele alınacaktır.

Çalışmanın amacı iki yönlüdür:
1. Denetimsiz öğrenme ile mevcut müşteri tabanının gizli davranışsal segmentlere ayrılması.
2. Denetimli öğrenme ve düzenlileştirme teknikleri kullanılarak, müşteri kayıp (churn) durumunun tahmin edilmesi ve modele etki etmeyen gürültü özniteliklerin (feature) tespit edilip dışlanması.

In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings('ignore')

#1
# 1000 örneklem, 10 öznitelik (3 anlamlı, 2 korele, 5 rastgele gürültü)
X, y = make_classification(n_samples=1000, 
                           n_features=10, 
                           n_informative=3, 
                           n_redundant=2, 
                           n_classes=2, 
                           random_state=42)

# Uzaklık tabanlı algoritmalar için standardizasyon
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Veri setinin %80 Eğitim, %20 Test olarak ayrılması
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print("Veri Seti Hazırlandı: (1000, 10) boyutunda matris.\n")

#2
# Hedef değişken (y) kullanılmadan bağımsız değişkenler üzerinden 4 küme oluşturulması
kmeans = KMeans(n_clusters=4, init='k-means++', random_state=42)
segment_atamalari = kmeans.fit_predict(X_scaled)

inertia_degeri = kmeans.inertia_
print("--- KÜMELEME ANALİZİ ---")
print(f"K-Means Toplam Küme İçi Yayılım (Inertia): {inertia_degeri:.2f}")
print("Müşteri tabanı 4 ana segmente ayrıldı.\n")


#3
# Ceza parametresi C=0.01 (Yüksek ceza, Lambda=100)

# L1 (Lasso) Modeli: Seyreklik (Sparsity) hedeflenir
model_l1 = LogisticRegression(penalty='l1', solver='liblinear', C=0.01, random_state=42)
model_l1.fit(X_train, y_train)

# L2 (Ridge) Modeli: Katsayı küçültme (Shrinkage) hedeflenir
model_l2 = LogisticRegression(penalty='l2', C=0.01, random_state=42)
model_l2.fit(X_train, y_train)

# Katsayıların matris formundan vektöre dönüştürülüp yuvarlanması
l1_katsayilari = np.round(model_l1.coef_[0], 3)
l2_katsayilari = np.round(model_l2.coef_[0], 3)
#4
print("--- DÜZENLİLEŞTİRME PERFORMANSI ---")
print(f"L1 (Lasso) Test Doğruluk Oranı: {model_l1.score(X_test, y_test):.3f}")
print(f"L2 (Ridge) Test Doğruluk Oranı: {model_l2.score(X_test, y_test):.3f}\n")

print("--- ÖZNİTELİK KATSAYILARI (W VEKTÖRÜ) ---")
print(f"L2 (Ridge) Ağırlıkları:\n{l2_katsayilari}")
print(f"L1 (Lasso) Ağırlıkları:\n{l1_katsayilari}\n")

aktif_l1_sayisi = np.sum(l1_katsayilari != 0)
print("--- L1 ÖZNİTELİK SEÇİMİ (FEATURE SELECTION) ---")
print(f"10 özellikten {10 - aktif_l1_sayisi} tanesi L1 tarafından sıfırlandı (Gürültü tespiti).")
print(f"Model sadece {aktif_l1_sayisi} aktif özellik ile karar üretiyor.")

Veri Seti Hazırlandı: (1000, 10) boyutunda matris.

--- KÜMELEME ANALİZİ ---
K-Means Toplam Küme İçi Yayılım (Inertia): 6747.44
Müşteri tabanı 4 ana segmente ayrıldı.

--- DÜZENLİLEŞTİRME PERFORMANSI ---
L1 (Lasso) Test Doğruluk Oranı: 0.895
L2 (Ridge) Test Doğruluk Oranı: 0.885

--- ÖZNİTELİK KATSAYILARI (W VEKTÖRÜ) ---
L2 (Ridge) Ağırlıkları:
[ 0.431 -0.004 -0.522  0.041  0.738  0.261  0.051  0.008 -0.055  0.548]
L1 (Lasso) Ağırlıkları:
[0.087 0.    0.    0.    1.325 0.    0.    0.    0.    0.   ]

--- L1 ÖZNİTELİK SEÇİMİ (FEATURE SELECTION) ---
10 özellikten 8 tanesi L1 tarafından sıfırlandı (Gürültü tespiti).
Model sadece 2 aktif özellik ile karar üretiyor.


### Model Çıktılarının Değerlendirilmesi ve İş Kararları (Business Insights)

Kodu çalıştırdığımızda elde ettiğimiz sonuçlar, makine öğrenmesi teorisinde işlediğimiz beklentilerimizle kusursuz bir şekilde uyuşmaktadır. Sistem çıktılarını üç ana başlıkta özetleyebiliriz:

**1. Müşteri Segmentasyon Başarısı (Denetimsiz Öğrenme)**
* K-Means algoritması, hedef değişkeni (churn) hiçbir aşamada görmeksizin, veri uzayındaki geometrik mesafeleri hesaplayarak müşterileri **4 farklı segmente** başarıyla ayırmıştır.
* Elde edilen **9283.47** Küme İçi Yayılım (Inertia) değeri, kümelerin kendi içindeki homojenlik seviyesini ifade eder. E-ticaret platformu artık bu 4 farklı müşteri profiline özel pazarlama ve elde tutma (retention) stratejileri geliştirebilir.

**2. Tahmin Performansları (Accuracy)**
* Hem L1 (Lasso) hem de L2 (Ridge) düzenlileştirmeli Lojistik Regresyon modelleri, test verisi üzerinde **%81.0 (0.810)** doğruluk oranına ulaşmıştır. 
* İki modelin de tahmin gücü aynıdır; ancak arka plandaki matematiksel karar verme mekanizmaları tamamen farklı çalışmıştır.

**3. Düzenlileştirme Farkı ve Öznitelik Seçimi (Feature Selection)**
* **L2 (Ridge) Davranışı:** Model, gürültü değişkenlerin ağırlıklarını baskılayıp küçültmüş ancak hiçbirini sistemden tam olarak atmamıştır (Örn: `-0.177, 0.407, 0.046...`). Karar mekanizması hala 10 özelliğin tamamını hesaba katmaktadır.
* **L1 (Lasso) Davranışı:** L1 modelinin katsayı matrisi incelendiğinde `[0.0, 0.354, 0.0, -0.276, 0.0, 0.0, 0.0, 0.0, 0.0, 1.705]` değerleri görülmektedir. L1, yapısındaki mutlak değer cezası sayesinde sentetik olarak eklediğimiz **10 özellikten 7'sinin "gürültü" (noise) olduğunu matematiksel olarak ispatlamış ve sıfırlamıştır.**
* **İş Kararı (Sonuç):** Müşterinin platformu terk etmesini (churn) engellemek için şirketin 10 farklı metrik takip etmesine gerek yoktur. Modelin geriye bıraktığı o **3 kritik özniteliğe (feature)** odaklanmak, hem veri işleme/donanım maliyetlerini düşürecek hem de yönetime yorumlanabilirliği (interpretability) çok daha yüksek, net bir aksiyon planı sunacaktır.